# Chapter 1 - Environment verification

This notebook reproduces the setup verification shown in Chapter 1, then verifies the chapter's four-layer GeoAI stack.

## Verify your setup - imports and clustering setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import scipy.ndimage
from sklearn.cluster import DBSCAN
from geoai_utils import *

## Seed, style, and device

In [2]:
seed_everything(42)
apply_style()
device = get_device()
print(f'Device: {device}')

Device: cuda


## Synthetic autocorrelated scene

In [3]:
SCENE_SIZE = 64
rng = np.random.default_rng(42)
noise = rng.standard_normal((SCENE_SIZE, SCENE_SIZE))
scene = scipy.ndimage.gaussian_filter(noise, sigma=5)
print(f'Scene shape: {scene.shape}   dtype: {scene.dtype}')
print(f'Value range: [{scene.min():+.2f}, {scene.max():+.2f}]')

Scene shape: (64, 64)   dtype: float64
Value range: [-0.15, +0.13]


## The chapter's four layers - live verification

Each row below is imported from the actual environment. A green `True` means the package loaded successfully.

In [4]:
import pandas as pd
from geoai_layers import verify_four_layers

layer_status = pd.DataFrame(verify_four_layers())
display(layer_status)
assert layer_status['status'].all(), 'At least one layer import failed'
print('All four layers verified successfully.')

,layer,package,version,status
0,Data,rasterio,1.4.4,True
1,Data,geopandas,1.1.4,True
2,Data,pandas,3.0.5,True
3,Data,shapely,2.1.2,True
4,Data,pyproj,3.7.2,True
5,Data,osmnx,2.1.1,True
6,Data,pystac-client,0.9.0,True
7,ML,torch,2.13.0+cu130,True
8,ML,scikit-learn,1.9.0,True
9,ML,segment-geospatial,1.4.2,True


All four layers verified successfully.


## GPU computation check

In [5]:
scene_tensor = torch.from_numpy(scene).float().to(device)
mean_square = scene_tensor.square().mean().item()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA runtime: {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Tensor device: {scene_tensor.device}')
print(f'Mean square: {mean_square:.6f}')

PyTorch: 2.13.0+cu130
CUDA runtime: 13.0
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
Tensor device: cuda:0
Mean square: 0.003926
